# Phase 8 — Sequential annealing via freeze / unfreeze

Demonstrates the *worst-actor* annealing recipe for pushing $|F|_{\max}$ below the joint-optimization plateau at $\phi=0.86$.

**The recipe per round:**
1. Identify the worst-balanced particle in the current state via `s.eval_force_balance()`
2. **Freeze** all other particles via `s.set_driven_particles(others, ZERO_TRAJ, frozen=True)` — the `s.fire()` / `s.lbfgs()` optimizers honor `driven_mask` + `shape_frozen` flags natively
3. Optimize the lone particle with `s.fire()` then `s.lbfgs()` — drives its solo residual to machine precision
4. **Unfreeze** all particles via `s.clear_driven_particles()`; joint-optimize the full system to its new plateau
5. Track best-of-all-rounds; only accept new state if global $|F|_{\max}$ improved

**Why this works (sometimes):** the joint-optimization plateau at $|F|_{\max}\!\sim\!2.5\!\times\!10^{-3}$ is set by *multi-particle frustration*, not single-particle physics. Anneal-then-rejoin can steer the joint optimizer into a deeper basin — but only in basin-compatible cases. Best-of tracking guards against the non-monotonic regressions observed in practice.

**Pre-requisite state file:** the notebook loads `../../tmp/phase8_stability/post_swell_state.npz`. If you don't have it, run `python_v2/notebooks/phase8_adam_quench.ipynb` first to generate a φ=0.86 starting state, or adjust `INPUT` below.

## 1. Setup

In [ ]:
import sys, os, time
sys.path.insert(0, '..')                                  # python_v2 root
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import numpy as np
import tensorflow as tf
import src.simulation.tf_sim as tfm; tfm.set_dtype(tf.float64)
from src.epd.particles import ParticleSpec
from src.epd.system import System
import matplotlib.pyplot as plt

P, N  = 16, 60
INPUT = '../../tmp/phase8_stability/post_swell_state.npz'   # post-swell at φ=0.86
ZERO_TRAJ = np.zeros(18, dtype=np.float64)                  # no prescribed motion

# Round budgets — demo-sized. For production, use 25k FIRE + 10k LBFGS joint.
SINGLE_FIRE   = 3_000
SINGLE_LBFGS  = 1_000
JOINT_FIRE    = 8_000
JOINT_LBFGS   = 3_000
N_ROUNDS      = 4

In [ ]:
# Build system, load post-swell state, zero all damping (pure quench)
def build_load():
    s = System(18.0, 18.0, periodic_x=True, periodic_y=True,
               dt_factor=0.25, candidacy_kind='prcm', E_candidates=9)
    s.add_particles(ParticleSpec(count=P, type='emulsion', gamma=1.0, kappa=0.02,
                                  N_nodes=N, Oh=5.0,
                                  poly_dist={'type':'bimodal','ratio':0.5,'delta':0.2}))
    s.initialize(phi_target=0.49, seed=42, verbose=False, relax_only=True, n_relax_init=0)
    s.restore_state(INPUT)
    DTYPE = s._params['_alpha_tf'].dtype
    s._params['xi_drag_per_p']    = tf.zeros([P], dtype=DTYPE)
    s._params['_alpha_tf']        = tf.constant(0.0, dtype=DTYPE)
    s._params['alpha_damp_per_p'] = tf.zeros([P], dtype=DTYPE)
    s.beta_rb = 0.0
    return s

def snapshot(s):
    return {'x_all': s._state['x_all'].numpy().copy(),
            'x_cm':  s._state['x_cm'].numpy().copy()}
def restore(s, snap):
    DTYPE = s._state['x_all'].dtype
    s._state['x_all'] = tf.constant(snap['x_all'], dtype=DTYPE)
    s._state['x_cm']  = tf.constant(snap['x_cm'],  dtype=DTYPE)

def per_particle_Fmax(s):
    """(P,) array of per-particle max nodal |F|. eval_force_balance()['top_stuck']
    only returns the top 10 *worst* particles — this returns all of them so we
    can read out the freshly-relaxed free particle's residual."""
    f = s.eval_forces()['f_total']
    return np.linalg.norm(f, axis=2).max(axis=1)

s = build_load()
print(f'φ = {s.phi_outer:.4f}   P = {P}   N_nodes = {N}   box = {s._params["box"].numpy()}')

## 2. The freeze API

The `System` class exposes per-particle constraint flags that the `s.run()`, `s.fire()`, `s.adam()`, and `s.lbfgs()` kernels all honor:

| flag combination | semantic | use case |
| --- | --- | --- |
| `driven_mask=1`, `shape_frozen=1`, `traj=0` | particle exactly fixed in space | this notebook |
| `driven_mask=1`, `shape_frozen=0`, `traj=0` | CM fixed, shape can deform | hold particle as anchor |
| `driven_mask=1`, `shape_frozen=1`, `traj≠0` | rigid-body prescribed motion | Couette outer/inner ring |
| both 0 | free particle | default |

Wired via two methods:

- `s.set_driven_particles(indices, traj_rows, frozen=True)` — flag the indices
- `s.clear_driven_particles()` — reset everyone to free

Note: for the optimizers (adam/fire/lbfgs), only the DC components of `traj` (`v_dc_x`, `v_dc_y`, `omega_orbit_dc`) are honored; AC and spin require true time integration via `s.run()`. `s.lbfgs()` doesn't advance prescribed motion at all (no natural step count).

## 3. Baseline joint optimization

Establish the all-particles-free plateau. This is the floor we're trying to beat.

In [ ]:
F_initial = s.eval_force_balance()['F_max']
print(f'Initial |F|_max (before any opt): {F_initial:.3e}\n')

print(f'Baseline joint opt: s.fire({JOINT_FIRE}) + s.lbfgs({JOINT_LBFGS})...')
t0 = time.time()
s.fire (JOINT_FIRE,  verbose=False)
s.lbfgs(JOINT_LBFGS, verbose=False)
F_baseline = s.eval_force_balance()['F_max']
print(f'  baseline |F|_max = {F_baseline:.3e}   [wall {time.time()-t0:.1f}s]')

baseline_snap = snapshot(s)
trajectory = [('baseline', F_baseline, -1, None)]

## 4. One annealing round

Verbose printing so you can see each step.

In [ ]:
# ── A) Identify the worst actor ──────────────────────────────────────────────
fb = s.eval_force_balance()
worst_p = int(fb['top_stuck'][0][0])
worst_F = fb['top_stuck'][0][2]
print(f'A)  worst actor:  p = {worst_p}   |F| = {worst_F:.3e}')
print(f'    top 3 stuck:  {[(int(p), f"{f:.2e}") for p, _, f in fb["top_stuck"][:3]]}')

# ── B) Freeze all others via the API, optimize the lone particle ─────────────
others = [p for p in range(P) if p != worst_p]
s.set_driven_particles(others, [ZERO_TRAJ]*len(others), frozen=True)
print(f'\nB)  freeze all except p={worst_p}, single-particle optimize...')
t0 = time.time()
s.fire (SINGLE_FIRE,  verbose=False)
s.lbfgs(SINGLE_LBFGS, verbose=False)
per_p = per_particle_Fmax(s)
F_solo = per_p[worst_p]                              # the freshly-relaxed particle
F_global_after_B = per_p.max()                       # dominated by frozen-others residuals
print(f'    p={worst_p} solo |F|_max = {F_solo:.3e}   '
      f'(global |F|_max = {F_global_after_B:.3e})   [wall {time.time()-t0:.1f}s]')

# ── C) Unfreeze everyone, joint optimize ─────────────────────────────────────
s.clear_driven_particles()
print(f'\nC)  unfreeze all, joint optimize...')
t0 = time.time()
s.fire (JOINT_FIRE,  verbose=False)
s.lbfgs(JOINT_LBFGS, verbose=False)
F_round1 = s.eval_force_balance()['F_max']
print(f'    joint plateau |F|_max = {F_round1:.3e}   [wall {time.time()-t0:.1f}s]')

trajectory.append(('round1', F_round1, worst_p, F_solo))
print(f'\n→  baseline {F_baseline:.3e}  →  round 1 {F_round1:.3e}  '
      f'({"IMPROVED" if F_round1 < F_baseline else "REGRESSED"})')

## 5. Multi-round loop with **best-of tracking**

Rounds 2-N. Key safeguard: after each round, only **accept** the new state if global $|F|_{\max}$ improved. If a round regressed (which empirically happens after 3-4 rounds), **restore the best-so-far state** and continue searching from there.

This addresses the V-shape regression observed in the standalone study: it locks in gains and prevents drift into a worse basin.

In [ ]:
best_snap = snapshot(s) if F_round1 < F_baseline else baseline_snap
best_F    = min(F_round1, F_baseline)
if F_round1 >= F_baseline:
    restore(s, baseline_snap)
    print(f'[round 1 regressed — restored baseline snapshot]')

for rnd in range(2, N_ROUNDS + 1):
    print(f'\n{"="*60}\nRound {rnd}    (current best: {best_F:.3e})\n{"="*60}')
    fb = s.eval_force_balance()
    worst_p = int(fb['top_stuck'][0][0])
    print(f'  worst actor: p={worst_p}  |F|={fb["top_stuck"][0][2]:.3e}')

    others = [p for p in range(P) if p != worst_p]
    s.set_driven_particles(others, [ZERO_TRAJ]*len(others), frozen=True)
    t0 = time.time()
    s.fire (SINGLE_FIRE,  verbose=False)
    s.lbfgs(SINGLE_LBFGS, verbose=False)
    F_solo = per_particle_Fmax(s)[worst_p]
    print(f'  solo |F|_max = {F_solo:.3e}   [B wall {time.time()-t0:.1f}s]')

    s.clear_driven_particles()
    t0 = time.time()
    s.fire (JOINT_FIRE,  verbose=False)
    s.lbfgs(JOINT_LBFGS, verbose=False)
    F_new = s.eval_force_balance()['F_max']
    print(f'  joint |F|_max = {F_new:.3e}   [C wall {time.time()-t0:.1f}s]')

    if F_new < best_F:
        best_F    = F_new
        best_snap = snapshot(s)
        verdict   = f'ACCEPT (new best: {best_F:.3e})'
    else:
        restore(s, best_snap)
        verdict   = f'REJECT (restored best: {best_F:.3e})'
    print(f'  → {verdict}')
    trajectory.append((f'round{rnd}', F_new, worst_p, F_solo))

print(f'\n{"="*60}')
print(f'Done.   baseline {F_baseline:.3e}   →   best {best_F:.3e}   ({F_baseline/best_F:.1f}× drop)')

## 6. Summary + plot

In [ ]:
print(f'{"phase":<10}  {"|F|_max":>11}  {"worst_p":>8}  {"solo |F|":>11}')
print('-' * 46)
for name, fmax, p, p_solo in trajectory:
    p_str = str(p) if p >= 0 else '-'
    p_solo_str = f'{p_solo:.3e}' if p_solo is not None else '-'
    print(f'{name:<10}  {fmax:>11.3e}  {p_str:>8}  {p_solo_str:>11}')

labels = [t[0] for t in trajectory]
fmaxes = np.array([t[1] for t in trajectory])
running_best = np.minimum.accumulate(fmaxes)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(range(len(labels)), fmaxes,       'b-o', lw=1.3, label='round outcome')
ax.semilogy(range(len(labels)), running_best, 'r--', lw=1.5, label='best-of (running min)')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30)
ax.set_ylabel('global |F|_max  (after joint opt)')
ax.grid(alpha=0.3, which='both'); ax.legend()
ax.set_title(f'Sequential annealing — {F_baseline/best_F:.1f}× drop from baseline')
plt.tight_layout(); plt.show()

## 7. Visualize the best-of state

In [ ]:
restore(s, best_snap)
fig, ax = plt.subplots(figsize=(7, 7))
s.set_color_palette('palette1', seed=1)
s.render(ax=ax)
ax.set_title(f'Best-of state   |F|_max = {best_F:.3e}   φ = {s.phi_outer:.4f}')
plt.show()

## Notes

**Adapting the freeze pattern to other use-cases:**

- *Free the worst K particles together*: instead of one `worst_p`, pass `others = [p for p in range(P) if p not in worst_K]` to `set_driven_particles`. Multi-particle solo solves may help when worst actors are mutually coupled.
- *Free a contiguous cluster*: feed `others` as everyone not in the cluster. Useful for shear-band ICs where the band geometry is known.
- *CM-fixed but shape-free*: `s.set_driven_particles(indices, [ZERO_TRAJ]*len(indices), frozen=False)` — particles' centers of mass don't move, but their nodes deform.

**Production budgets:**

| Stage | Demo (this notebook) | Production |
| --- | --- | --- |
| Single FIRE / LBFGS | 3k / 1k | 5k / 2k |
| Joint FIRE / LBFGS | 8k / 3k | 25k / 10k |
| N_ROUNDS | 4 | 5-8 (with best-of) |

**Empirical observations from the phase-8 anneal study** (at φ=0.86, P=16):
- Single-particle solo always reaches machine precision (1e-11 to 1e-13) regardless of which particle.
- Joint plateau improves for 2-3 rounds (up to ~3× from baseline), then can regress.
- Best-of tracking locks in the gain — without it, you can end up worse than baseline.
- The improvement is real (different basin of the marginal-jamming landscape), not numerical.

**See also:** `papers/summary_of_methods/main.tex` — the perimeter-regularization $k_{\rm reg}$ pseudo-force discretization section discusses why the joint plateau exists in the first place.